In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import itertools
import numpy as np

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx
lp = sjx.optical_depth.line_profile

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('default')
plt.close('all')
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
mpl.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
xx = jnp.linspace(-9, 9, 2**13)

sigma_low = 1e-3
sigma_high = 1.0
gamma_low = 1e-3
gamma_high = 1.0

params = jnp.array([
    (sigma_low, gamma_low),
    (sigma_high, gamma_low),
    (sigma_low, gamma_high),
    (sigma_high, gamma_high)
])

Line_profiles = {}

Line_profiles['scipy_wofz'] = lp.Voigt(xx, params[:, 0], params[:, 1], mode="scipy_wofz")

voigt_convolution_batched = jit(
    vmap(lambda s, g: lp.Voigt(xx, s, g, mode="convolution", oversample=4), in_axes=(0, 0))
)
Line_profiles["JAX_convolution"] = voigt_convolution_batched(params[:, 0], params[:, 1])

voigt_exojax_batched = jit(
    vmap(lambda s, g: lp.Voigt(xx, s, g, mode="ExoJAX"), in_axes=(0, 0))
)
Line_profiles["exojax"] = voigt_exojax_batched(params[:, 0], params[:, 1])

In [ ]:
P = len(params)
colors = np.array(plt.cm.get_cmap("tab10").colors[:P])

fig, (ax, ax_res) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw=dict(height_ratios=[2, 1]))

for ii in range(P):
    ax.plot(xx, Line_profiles['scipy_wofz'][ii], color=colors[ii], linestyle="-", linewidth=1.0)
    ax.plot(xx, Line_profiles['JAX_convolution'][ii], color=colors[ii], linestyle="--", linewidth=2.)
    ax.plot(xx, Line_profiles['exojax'][ii], color=colors[ii], linestyle=":", linewidth=3.5)

for ii in range(P):
    ax_res.plot(
        xx, Line_profiles['JAX_convolution'][ii] / Line_profiles['scipy_wofz'][ii] - 1,
        color=colors[ii], linestyle="--", linewidth=2.0
    )
    ax_res.plot(
        xx, Line_profiles['exojax'][ii] / Line_profiles['scipy_wofz'][ii] - 1,
        color=colors[ii], linestyle=":", linewidth=3.5
    )

method_handles = [
    Line2D([0], [0], color="k", lw=8,  ls="-",  label="Scipy wofz"),
    Line2D([0], [0], color="k", lw=4,  ls="--", label="JAX convolution"),
    Line2D([0], [0], color="k", lw=2,  ls=":",  label="exojax"),
]
leg1 = ax.legend(handles=method_handles, loc="upper right", fontsize=16)
ax.add_artist(leg1)

param_handles = []
for ii in range(P):
    sigma, gamma = params[ii]
    lbl = rf"$\sigma={sigma:g},\ \gamma={gamma:g}$"
    param_handles.append(Line2D([0], [0], color=colors[ii], lw=6, label=lbl))

leg2 = ax.legend(handles=param_handles, loc="upper left", fontsize=16)
ax.add_artist(leg2)

ax.set_title("Voigt profiles")
ax.set_ylabel(r"$V(x;\ \sigma,\ \gamma)$")
ax.set_yscale("log")
ax.set_ylim(1e-6, 1e3)

ax_res.axhline(0.0, color="k", linewidth=1, alpha=0.5)
ax_res.set_xlabel("x")        # only bottom panel gets the x-label
ax_res.set_ylabel("residual")
ax_res.set_yscale("symlog", linthresh=1e-8)

major_ticks = [-1e-2, -1e-6, 0.0, 1e-6, 1e-2]
ax_res.set_yticks(major_ticks)

ax_res.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
ax_res.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

plt.subplots_adjust(hspace=0.05)
fig.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------
# Grid and parameters
# -----------------------------------------------
xx = jnp.linspace(-9.0, 9.0, 2**12)  # frequency (or velocity) offset grid

sigma0 = 0.2
gamma0 = 0.5

# -----------------------------------------------
# Helpers: scalar wrapper using the Voigt wrapper (ExoJAX mode)
# -----------------------------------------------
def voigt_exojax_scalar(x, sigma, gamma):
    """
    Scalar wrapper around the Voigt wrapper in ExoJAX mode
    so that we can use grad in the same style as the exojax example.
    """
    # lp.Voigt returns an array over x; take the single element
    return lp.Voigt(jnp.atleast_1d(x), sigma, gamma, mode="ExoJAX")[0]

# Derivative wrt x (ν)
dVoigt_dx_exojax = jit(
    vmap(grad(voigt_exojax_scalar, argnums=0), (0, None, None), 0)
)

# Derivative wrt sigma (Gaussian width)
dVoigt_dsigma_exojax = jit(
    vmap(grad(voigt_exojax_scalar, argnums=1), (0, None, None), 0)
)

# Base Voigt profile from ExoJAX implementation via wrapper
Voigt_exojax_profile = jit(lambda x, s, g: lp.Voigt(x, s, g, mode="ExoJAX"))(
    xx, sigma0, gamma0
)

dV_dx_exojax = dVoigt_dx_exojax(xx, sigma0, gamma0)
dV_dsigma_exojax = dVoigt_dsigma_exojax(xx, sigma0, gamma0)

# -----------------------------------------------
# Autodiff through the JAX convolution Voigt (via wrapper)
# -----------------------------------------------
def voigt_conv_profile(sigma, gamma):
    """
    Voigt profile computed via JAX convolution on a fixed grid xx.
    This is fully JAX and supports autodiff.
    """
    return lp.Voigt(xx, sigma, gamma, mode="convolution", oversample=4)

# Profile for our reference parameters
Voigt_conv_profile = jit(voigt_conv_profile)(sigma0, gamma0)

# Derivative wrt sigma for the *entire profile* using jacfwd
dVoigtConv_dsigma = jit(jacfwd(lambda s: voigt_conv_profile(s, gamma0)))
dVConv_dsigma = dVoigtConv_dsigma(sigma0)  # shape (xx.size,)

# If you also want derivative wrt gamma:
dVoigtConv_dgamma = jit(jacfwd(lambda g: voigt_conv_profile(sigma0, g)))
dVConv_dgamma = dVoigtConv_dgamma(gamma0)

# -----------------------------------------------
# Optionally: use grad for a scalar functional of the profile
# (e.g. value at line center or integrated area)
# -----------------------------------------------
def voigt_conv_at_center(sigma, gamma):
    """Value of the convolved Voigt at x=0 (line center)."""
    prof = voigt_conv_profile(sigma, gamma)
    # xx is symmetric around 0; pick closest index to zero
    idx0 = xx.size // 2
    return prof[idx0]

d_center_dsigma = jit(grad(voigt_conv_at_center, argnums=0))(sigma0, gamma0)
d_center_dgamma = jit(grad(voigt_conv_at_center, argnums=1))(sigma0, gamma0)

print("d V_conv(x=0) / d sigma =", float(d_center_dsigma))
print("d V_conv(x=0) / d gamma =", float(d_center_dgamma))

# -----------------------------------------------
# Plotting example
# -----------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True)

ax = axes[0, 0]
ax.plot(xx, Voigt_exojax_profile, label="Voigt (ExoJAX mode)")
ax.plot(xx, Voigt_conv_profile, linestyle="--", label="Voigt (convolution mode)")
ax.set_ylabel("V(x)")
ax.set_title("Voigt profiles")
ax.legend()

ax = axes[1, 0]
ax.plot(xx, dV_dx_exojax, label=r"$\partial_x V_{\mathrm{ExoJAX}}$")
ax.set_xlabel("x")
ax.set_ylabel(r"dV/dx")
ax.set_title("Derivative wrt x (ExoJAX mode)")
ax.legend()

ax = axes[0, 1]
ax.plot(xx, dV_dsigma_exojax, label=r"$\partial_\sigma V_{\mathrm{ExoJAX}}$")
ax.plot(xx, dVConv_dsigma, linestyle="--", label=r"$\partial_\sigma V_{\mathrm{conv}}$")
ax.set_ylabel(r"dV/dσ")
ax.set_title("Derivative wrt σ")
ax.legend()

ax = axes[1, 1]
ax.plot(xx, dVConv_dgamma, label=r"$\partial_\gamma V_{\mathrm{conv}}$")
ax.set_xlabel("x")
ax.set_ylabel(r"dV/dγ")
ax.set_title("Derivative wrt γ (convolution)")
ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
from skewerjax.constants.lines import LINES

In [ ]:
load_absorption_lines = ["HI-LyA", "HI-LyB", "CIV-1551"]
absorption_lines = [LINES[lid] for lid in load_absorption_lines]
N_sp = len(absorption_lines) # number of absroption lines ''species'' (considering different absorption lines from different species at the same time)

# generate synthetic data for different skerwers
N_sk = 7                                                                              # number of skerwers
N_ppsk = 1                                                                            # number of points sampled per skerwer
tau_coll_ns = jnp.inf * jnp.ones((N_sp, N_sk, N_ppsk))                                # UNITS: ns
b_turb_m_per_s = jnp.zeros((N_sp, N_sk, N_ppsk))                                      # UNITS: m/s
T_K = jnp.ones((N_sp, N_sk, N_ppsk))*jnp.geomspace(0.001, 1000, N_sk)[None, :, None] # UNITS: K

# generate sampled wavelengths
N_lambda = 2**18                                                 # number of sampled wavelengths
lambda_min = min([line.lambda0_A for line in absorption_lines]) - 300 # [A]
lambda_max = max([line.lambda0_A for line in absorption_lines]) + 300 # [A]
lambdas = jnp.linspace(lambda_min, lambda_max, N_lambda)         # (N_lambda,)



line_profiles_batched = jit(
    lambda lambdas, T_K, b_turb_m_per_s, tau_coll_ns: lp.line_profiles(
        lambdas=lambdas, T_K=T_K, b_turb_m_per_s=b_turb_m_per_s,
        tau_coll_ns=tau_coll_ns, absorption_lines=absorption_lines, mode="ExoJAX",
    )
)
phi = line_profiles_batched(lambdas, T_K, b_turb_m_per_s, tau_coll_ns)

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(lambdas)           # (N_lambda,)
phi_np = np.asarray(phi)          # (N_sp, N_sk, N_ppsk, N_lambda)
T_vals = np.asarray(T_K[0, :, 0])          # (N_sk,)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])
linestyles = ['-', '--', ':']


fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(absorption_lines):
    ax.axvline(line.lambda0_A, color="k", linestyle=":", linewidth=1.0, alpha=0.4)
    ls = linestyles[i_sp % len(linestyles)]
    for i_sk in range(N_sk):
        profile = phi_np[i_sp, i_sk, 0, :]   # full wavelength range
        ax.plot(lambdas_np, profile, color=colors[i_sk], linestyle=ls, linewidth=2.0)

# Axes formatting
ax.set_yscale("log")
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\phi(\lambda)$")
ax.grid(alpha=0.2, which="both")
ax.set_title("Line profiles for different transitions and temperatures", fontsize=20)

# Legend for temperatures (colors)
temp_handles = []
for i_sk in range(N_sk):
    lbl = rf"$T = {T_vals[i_sk]:.2g}\,\mathrm{{K}}$"
    temp_handles.append(Line2D([0], [0], color=colors[i_sk], lw=4, linestyle="-", label=lbl))
leg_T = ax.legend(handles=temp_handles, loc="upper left", fontsize=16, ncols=1)
ax.add_artist(leg_T)

# Legend for lines (linestyles)
line_handles = []
for i_sp, line in enumerate(absorption_lines):
    ls = linestyles[i_sp % len(linestyles)]
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    line_handles.append(Line2D([0], [0], color="k", lw=3, linestyle=ls, label=lbl))
leg_L = ax.legend(handles=line_handles, loc="lower right", fontsize=16, ncol=N_sp)

fig.tight_layout()
plt.show()



# Zoom half-width around each line center [Angstrom]
delta_lambda_A = 6.0   # adjust to taste
fig, axes = plt.subplots(nrows=1, ncols=N_sp, figsize=(14, 6), sharex=False, sharey=True)
if N_sp == 1:
    axes = [axes]  # in case N_sp=1, make it iterable

for i_sp, (ax, line) in enumerate(zip(axes, absorption_lines)):
    lambda0 = line.lambda0_A
    ax.axvline(lambda0, color="k", linestyle="-.", linewidth=2.0)

    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]  # (N_lambda_window,)

    # Plot each temperature (each skewer)
    for i_sk in range(N_sk):
        profile = phi_np[i_sp, i_sk, 0, mask]  # (N_lambda_window,)
        ax.plot(lam_win, profile, color=colors[i_sk], linestyle=linestyles[i_sp], linewidth=2.0)

    # Panel title and scaling
    ax.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax.set_yscale("log")
    ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax.grid(alpha=0.2, which="both")
axes[0].set_ylabel(r"$\phi(\lambda)$")

# Temperature legend (shared across panels)
temp_handles = []
for i_sk in range(N_sk):
    lbl = rf"$T = {T_vals[i_sk]:.2g}\,\mathrm{{K}}$"
    temp_handles.append(Line2D([0], [0], color=colors[i_sk], lw=5, label=lbl))
leg = axes[0].legend(handles=temp_handles, loc="lower left", fontsize=14, ncol=2)
axes[0].add_artist(leg)

fig.tight_layout()
plt.show()